# Day 09 — Lê Quang Khải, 2A202603237

Chạy bài đã hoàn thành từ ZIP. Chọn **Runtime → Change runtime type → T4 GPU** trước khi chạy. Clean đang **Hold**, chỉ là bản nháp để thử nghiệm. Notebook gọi script train gốc; không tạo kết quả trước khi train.


In [ ]:
import subprocess, sys
subprocess.run(['nvidia-smi', '-L'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.214', 'imagehash', 'pyyaml', 'matplotlib'], check=True)


## Upload bài đã làm

Chọn `day09-lab-2A202603237.zip` nằm cạnh thư mục `lab/` trên máy bạn. Dữ liệu được giải nén vào `/content/day09_lab`.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith('.zip')]
assert len(archives) == 1, 'Chọn đúng một ZIP bài lab.'
with zipfile.ZipFile(archives[0]) as archive:
    archive.extractall('/content')
os.chdir('/content/day09_lab')
assert Path('scripts/train_compare.py').exists()
import json
manifest = json.loads(Path('submission/export/clean_v1/manifest.json').read_text())
print('Clean:', manifest['n_images'], 'ảnh; quyết định:', manifest['decision']['decision'])


## Train và đo

Raw/clean dùng cùng yolo26n, 20 epoch, batch 16, ảnh 640, seed 0/1, gold_val và test. Cảnh báo nhãn hỏng ở raw là dữ liệu của thí nghiệm; không tự sửa raw. Chờ cả bốn lần chạy xong.


In [ ]:
subprocess.run([sys.executable, 'scripts/train_compare.py', '--variants', 'raw', 'clean', '--seeds', '0', '1', '--epochs', '20', '--device', '0', '--skip-existing'], check=True)
subprocess.run([sys.executable, 'scripts/summarize_results.py'], check=True)
from IPython.display import display, Markdown, Image
display(Markdown(Path('submission/results/comparison.md').read_text()))
display(Image(filename='submission/results/comparison.png'))


## Tải kết quả thật

Giải nén ZIP nhận được, chép `submission/results/*` về repo trên máy. Cập nhật REPORT.md mục 5 theo số liệu và chạy `make check`. Không thay đổi export/config sau khi train; nếu thay đổi phải train lại.


In [ ]:
import shutil
archive_path = shutil.make_archive('/content/day09_results_2A202603237', 'zip', root_dir='.', base_dir='submission/results')
files.download(archive_path)
